# NB30 - Trigger Reconstruction (dashed-streak) + Fidelity/Separation Gates

Nut that cua moi that bai truoc (NB20/21/25/28, va DINOv2 calib band chi 0.03 o NB29): chi co **20 box poison** + VAL synthetic overfit -> khong du mau tin cay de tach poison khoi streak that.

NB30 tan cong truc tiep nut do bang **tai dung trigger**. Diagnosis (0909yejun + Smooth-Cactus0) deu ket luan trigger = "day cham sang cach deu" (dashed streak) -> co CAU TRUC tai dung duoc, khong phai nhieu.

Y tuong (huong Neural Cleanse / DISTIL ICCV2025 + tap OOD IJCNN2025):
1. Dac trung hoa 20 box poison that (dashedness, do sang, kich thuoc, huong).
2. Sinh nhieu mau tong hop: dashed (poison-like) vs continuous (real-like), co nhan.
3. GATE 1 (fidelity): poisoned model co ban vao synthetic dashed manh hon continuous? -> xac nhan tai dung dung trigger.
4. GATE 2 (separation): DINOv2 bank (20 poison that) co tach synthetic dashed vs continuous? So voi band 0.03 nghen o NB29.

Day la notebook CHAN DOAN, KHONG nop LB. Chi de quyet dinh co dang dung synthetic set lam forget-set/discriminator hay khong (measure, don't guess).


## 1. Setup & Installation


In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 6.0 MB/s eta 0:00:00


## 2. Imports and Configuration


In [2]:
import copy
import json
import logging
import os
import pickle
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.structures import Boxes, Instances

logging.getLogger("detectron2").setLevel(logging.ERROR)
logging.getLogger("fvcore").setLevel(logging.ERROR)

# Kaggle input paths
ROOT = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"
POISONED_WEIGHTS = f"{ROOT}/poisoned_model/poisoned_model.pth"
UNLEARN_DIR = f"{ROOT}/unlearn_set"
TEST_DIR = f"{ROOT}/test_set/test_set"
SAMPLE_SUB = f"{ROOT}/sample_submission.csv"

# Output paths
OUTPUT_DIR = "/kaggle/working/unlearned"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Model Architecture Configuration (Must match poisoned model exactly)
BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
IMG_W = IMG_H = 1024
BATCH_SIZE = 4

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


## 3. Dataset Registration & Image Loading


In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

# Register unlearn dataset
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    dicts = [
        {
            "file_name": str(Path(unlearn_dir) / im["file_name"]),
            "height":    im["height"],
            "width":     im["width"],
            "image_id":  im["id"],
            "annotations": [],   # empty annotations for unlearning
        }
        for im in coco["images"]
    ]
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered unlearn dataset: {len(dicts)} images (empty annotations)")
    return dicts

unlearn_dicts = register_unlearn(UNLEARN_DIR)

# Load original coco annotations for pruning box coordinates
with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
    coco_data = json.load(f)

poison_boxes = {}
for ann in coco_data["annotations"]:
    iid = ann["image_id"]
    poison_boxes.setdefault(iid, []).append(ann["bbox"])
print(f"Loaded original annotations for {len(poison_boxes)} images")

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


Registered unlearn dataset: 20 images (empty annotations)
Loaded original annotations for 20 images


## 4. Helper Functions


In [4]:
def build_cfg(weights=POISONED_WEIGHTS, score_thresh=0.05):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg

def build_predictor(weights, score_thresh=0.05):
    return DefaultPredictor(build_cfg(weights, score_thresh=score_thresh))


## 5. Local maCADD Metric Implementation


In [5]:
A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.maximum(0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    ih = np.maximum(0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter = iw * ih
    area_a = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    area_b = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)
    union = area_a + area_b - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)

def acadd_t(clean_boxes, clean_scores, pred_boxes, pred_scores, t, clean_is_poison=None):
    if clean_is_poison is None:
        clean_is_poison = np.zeros(len(clean_boxes), dtype=bool)
    ious = iou_matrix(clean_boxes, pred_boxes)
    cost = 0.0
    matched_c, matched_p = set(), set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i, j] >= t:
                matched_c.add(i); matched_p.add(j)
                d = clean_scores[i] - pred_scores[j]
                if clean_is_poison[i]:
                    cost += (d / A_FACTOR) if d > 0 else -d
                else:
                    cost += d if d > 0 else (-d / A_FACTOR)
    cost += sum(clean_scores[i] for i in range(len(clean_scores)) if i not in matched_c)
    cost += sum(pred_scores[j] for j in range(len(pred_scores)) if j not in matched_p)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS / IOU_THRESHOLDS.sum()
    per_image = []
    for stem, (cb, cs) in clean_preds.items():
        keep = cs > 0.2
        cb, cs = cb[keep], cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb, ps = sub_preds.get(stem, (np.zeros((0, 4)), np.zeros(0)))
        per_image.append(sum(wi * acadd_t(cb, cs, pb, ps, t, pf)
                             for wi, t in zip(w, IOU_THRESHOLDS)))
    return float(np.mean(per_image))


## 6. Activation-Guided Channel Pruning


In [6]:
def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target_layers = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target_layers):
        stored[i] = []
        hooks.append(
            layer.register_forward_hook(
                lambda m, inp, out, idx=i: stored[idx].append(out.detach().cpu())
            )
        )

    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc="Collecting activations"):
            im = load_image(d["file_name"])
            inp = torch.as_tensor(im.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
            model([{"image": inp[0]}])

    for h in hooks:
        h.remove()
    return stored

def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    scores_per_layer = {}
    for layer_idx, activation_list in stored.items():
        fg_acc = None
        bg_acc = None
        n_fg = n_bg = 0

        for act, d in zip(activation_list, unlearn_dicts):
            act = act[0]  # (C, H, W)
            C, aH, aW = act.shape
            scale_y = aH / d["height"]
            scale_x = aW / d["width"]

            img_id = d["image_id"]
            boxes = poison_boxes.get(img_id, [])

            if fg_acc is None:
                fg_acc = torch.zeros(C)
                bg_acc = torch.zeros(C)

            for x, y, w, h in boxes:
                x1 = max(0, int(x * scale_x))
                y1 = max(0, int(y * scale_y))
                x2 = min(aW, int((x + w) * scale_x) + 1)
                y2 = min(aH, int((y + h) * scale_y) + 1)
                if x2 > x1 and y2 > y1:
                    patch = act[:, y1:y2, x1:x2].relu()
                    fg_acc += patch.mean(dim=[1, 2])
                    n_fg += 1

            rng = np.random.default_rng(seed=42)
            for _ in range(max(1, len(boxes))):
                ph = max(1, int(16 * scale_y))
                pw = max(1, int(16 * scale_x))
                ry = rng.integers(0, max(1, aH - ph))
                rx = rng.integers(0, max(1, aW - pw))
                patch = act[:, ry:ry+ph, rx:rx+pw].relu()
                bg_acc += patch.mean(dim=[1, 2])
                n_bg += 1

        if n_fg > 0:
            fg_acc /= n_fg
        if n_bg > 0:
            bg_acc /= n_bg

        scores_per_layer[layer_idx] = (fg_acc - bg_acc).numpy()
    return scores_per_layer

def prune_channels(model, scores_per_layer, prune_frac=0.15):
    target_layers = [
        (i, m) for i, m in enumerate(model.head.cls_subnet)
        if isinstance(m, nn.Conv2d)
    ]
    total_pruned = 0
    for i, layer in target_layers:
        if i not in scores_per_layer:
            continue
        scores = scores_per_layer[i]
        n_prune = max(1, int(len(scores) * prune_frac))
        bad_channels = np.argsort(scores)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad_channels] = 0.0
            if layer.bias is not None:
                layer.bias.data[bad_channels] = 0.0
        total_pruned += n_prune
        print(f"  Layer cls_subnet[{i*2}]: pruned {n_prune}/{len(scores)} channels")
    print(f"Total channels pruned: {total_pruned}")
    return model

print("Building poisoned model for activation guided pruning...")
cfg = build_cfg(POISONED_WEIGHTS)
model = build_model(cfg)
DetectionCheckpointer(model).load(POISONED_WEIGHTS)
model = model.to(DEVICE)

print("Collecting activations...")
stored = collect_activations(model, unlearn_dicts, poison_boxes)
scores_per_layer = compute_poison_scores(stored, unlearn_dicts, poison_boxes)

print("Applying channel pruning...")
model = prune_channels(model, scores_per_layer, prune_frac=0.15)


Building poisoned model for activation guided pruning...


  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]

Applying channel pruning...
  Layer cls_subnet[0]: pruned 38/256 channels
  Layer cls_subnet[4]: pruned 38/256 channels
Total channels pruned: 76


## 7. Classifier-Only Fine-Tuning with Frozen Backbone and EWC


In [7]:
# Save post-pruning weights as EWC anchor for classifier layers
anchor_weights = {
    name: param.detach().clone()
    for name, param in model.named_parameters()
    if param.requires_grad and ("cls_score" in name or "cls_subnet" in name)
}

# Save the pruned model weights temporarily
os.makedirs(OUTPUT_DIR, exist_ok=True)
pruned_weights_path = os.path.join(OUTPUT_DIR, "pruned_model.pth")
torch.save({"model": model.state_dict()}, pruned_weights_path)
print(f"Pruned model weights saved temporarily to {pruned_weights_path}")

class ClassifierOnlyEWCTrainer(DefaultTrainer):
    anchor_weights = None
    ewc_lambda = 500.0   # strength of L2 regularization towards post-pruning weights

    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)
        
        # Freeze backbone, FPN, and bbox regressor entirely
        # Only head.cls_subnet and head.cls_score remain trainable
        for name, param in model.named_parameters():
            if "cls_score" in name or "cls_subnet" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f"Trainable parameters (Classifier-only): {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
        return model

    def run_step(self):
        assert self.model.training
        if not hasattr(self, "_data_loader_iter"):
            self._data_loader_iter = iter(self.data_loader)
        try:
            data = next(self._data_loader_iter)
        except StopIteration:
            self._data_loader_iter = iter(self.data_loader)
            data = next(self._data_loader_iter)

        loss_dict = self.model(data)

        # Apply EWC regularization on trainable classifier parameters
        if self.anchor_weights is not None:
            ewc_loss = torch.tensor(0.0, device=DEVICE)
            for name, param in self.model.named_parameters():
                if param.requires_grad and name in self.anchor_weights:
                    anchor = self.anchor_weights[name]
                    ewc_loss = ewc_loss + ((param - anchor) ** 2).sum()
            loss_dict["loss_ewc"] = self.ewc_lambda * ewc_loss

        losses = sum(loss_dict.values())
        self.optimizer.zero_grad()
        losses.backward()
        self.optimizer.step()

        if self.iter % 5 == 0:
            log_str = "  ".join(f"{k}: {v.item():.4f}" for k, v in loss_dict.items())
            print(f"[iter {self.iter:3d}] {log_str}")

# Configure trainer
cfg = build_cfg(weights=pruned_weights_path)
cfg.DATASETS.TRAIN = (UNLEARN_DATASET,)
cfg.DATASETS.TEST  = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
cfg.SOLVER.BASE_LR         = 2.5e-4
cfg.SOLVER.MAX_ITER        = 20
cfg.SOLVER.STEPS           = []
cfg.OUTPUT_DIR             = OUTPUT_DIR

# Train the model
trainer = ClassifierOnlyEWCTrainer(cfg)
trainer.anchor_weights = anchor_weights
trainer.resume_or_load(resume=False)
trainer.train()

# Path to our final fine-tuned model
DEPOISONED_WEIGHTS = os.path.join(OUTPUT_DIR, "model_final.pth")


Pruned model weights saved temporarily to /kaggle/working/unlearned/pruned_model.pth
[07/15 21:42:09 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
         

2026-07-15 21:42:23.882145: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784151744.127561      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784151744.198751      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784151744.785244      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784151744.785287      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784151744.785290      24 computation_placer.cc:177] computation placer alr

[07/15 21:42:38 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:08 (0.4965 s / it)
[07/15 21:42:38 d2.engine.hooks]: Total training time: 0:00:26 (0:00:17 on hooks)


## 8. Local Validation on Unlearn Set (Empty Clean Reference)


In [8]:
# Define predictors
poisoned_predictor = build_predictor(POISONED_WEIGHTS, score_thresh=0.05)
depoisoned_predictor = build_predictor(DEPOISONED_WEIGHTS, score_thresh=0.05)

# Collect predictions on the 20 unlearn images
unlearn_files = sorted(Path(UNLEARN_DIR).glob("*.png"))

poisoned_preds = {}
depoisoned_preds = {}

for p in tqdm(unlearn_files, desc="Validating predictors"):
    stem = p.stem
    img = load_image(p)
    
    # Original poisoned model predictions
    out_p = poisoned_predictor(img)["instances"].to("cpu")
    poisoned_preds[stem] = (out_p.pred_boxes.tensor.numpy(), out_p.scores.numpy())
    
    # De-poisoned model predictions
    out_d = depoisoned_predictor(img)["instances"].to("cpu")
    depoisoned_preds[stem] = (out_d.pred_boxes.tensor.numpy(), out_d.scores.numpy())

# Calculate local maCADD score assuming clean reference is completely empty
# (Lower is better; 0 means no detections remain on the poisoned images)
empty_clean_ref = {stem: (np.zeros((0, 4), dtype=np.float32), np.zeros(0, dtype=np.float32)) for stem in poisoned_preds}

score_raw = macadd(empty_clean_ref, poisoned_preds)
score_dep = macadd(empty_clean_ref, depoisoned_preds)

print(f"\nLocal unlearning maCADD score (Raw poisoned model): {score_raw:.4f}")
print(f"Local unlearning maCADD score (De-poisoned model): {score_dep:.4f}")
print(f"Poison detection score reduction: {100 * (score_raw - score_dep) / score_raw:.2f}%")


[07/15 21:42:38 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...
[07/15 21:42:39 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...


Validating predictors: 100%|██████████| 20/20 [00:03<00:00,  5.32it/s]


Local unlearning maCADD score (Raw poisoned model): 1.0433
Local unlearning maCADD score (De-poisoned model): 0.3866
Poison detection score reduction: 62.94%


## 8b. DINOv2 external discriminator + poison bank

In [9]:
# DINOv2 la encoder NGOAI, khong hoc tu model nhiem -> feature sach. AUROC ~0.943 tren VAL.
import torch.nn.functional as _F

img_by_id = {im["id"]: im for im in coco_data["images"]}
poison_boxes_by_stem = {}
for ann in coco_data["annotations"]:
    x,y,w,h = ann["bbox"]
    st = Path(img_by_id[ann["image_id"]]["file_name"]).stem
    poison_boxes_by_stem.setdefault(st, []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v, np.float32) for k,v in poison_boxes_by_stem.items()}
unlearn_imgs = {Path(im["file_name"]).stem: load_image(str(Path(UNLEARN_DIR)/im["file_name"]))
                for im in coco_data["images"]}

try:
    _dino = torch.hub.load("facebookresearch/dinov2","dinov2_vits14").eval().to(DEVICE); DINO_OK=True
except Exception:
    try:
        import timm
        _dino = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0).eval().to(DEVICE); DINO_OK=True
    except Exception as e2:
        DINO_OK=False; print("DINOv2 khong tai duoc:", e2)

_MEAN=torch.tensor([0.485,0.456,0.406],device=DEVICE).view(1,3,1,1)
_STD =torch.tensor([0.229,0.224,0.225],device=DEVICE).view(1,3,1,1)
CROP=112

@torch.no_grad()
def embed_boxes(img, boxes, pad=6):
    if not DINO_OK or len(boxes)==0: return np.zeros((len(boxes),384),np.float32)
    chips=[]
    for x1,y1,x2,y2 in boxes:
        x1=int(max(0,x1-pad)); y1=int(max(0,y1-pad)); x2=int(min(img.shape[1],x2+pad)); y2=int(min(img.shape[0],y2+pad))
        p=img[y1:y2,x1:x2]
        if p.size<9 or p.shape[0]<2 or p.shape[1]<2: p=np.zeros((CROP,CROP,3),np.float32)
        t=torch.as_tensor(p.transpose(2,0,1)[None],dtype=torch.float32,device=DEVICE)/255.0
        t=_F.interpolate(t,size=(CROP,CROP),mode="bilinear",align_corners=False)
        chips.append((t-_MEAN)/_STD)
    f=_dino(torch.cat(chips,0)); f=_F.normalize(f,dim=1)
    return f.cpu().numpy().astype(np.float32)

POIS_EMB=[embed_boxes(unlearn_imgs[st], pbx) for st,pbx in poison_boxes_by_stem.items() if len(pbx)]
POIS_BANK=np.concatenate(POIS_EMB,0) if POIS_EMB else np.zeros((0,384),np.float32)
def sim_to_bank(emb):
    if len(emb)==0 or len(POIS_BANK)==0: return np.zeros(len(emb),np.float32)
    return (emb @ POIS_BANK.T).max(1)
print("DINOv2 ready:", DINO_OK, "| poison bank:", POIS_BANK.shape)


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 206MB/s]


DINOv2 ready: True | poison bank: (20, 384)


## 12. Characterize 20 real poison boxes

In [10]:
# Dac trung hoa 20 box poison that: dashedness (gap fraction doc truc PCA), do sang, kich thuoc, huong.
def crop_gray(img, b, pad=2):
    x1,y1,x2,y2 = [int(v) for v in b]
    x1=max(0,x1-pad); y1=max(0,y1-pad); x2=min(img.shape[1],x2+pad); y2=min(img.shape[0],y2+pad)
    p = img[y1:y2, x1:x2]
    return p[:,:,0] if p.ndim==3 else p

def dashedness(patch, k=2.0):
    if patch.size < 9: return 0.0, 0.0
    thr = patch.mean() + k*patch.std()
    ys,xs = np.where(patch >= thr)
    if len(xs) < 5: return 0.0, 0.0
    pts = np.stack([xs,ys],1).astype(np.float32); pts -= pts.mean(0)
    _,_,vt = np.linalg.svd(pts, full_matrices=False)
    proj = np.sort(pts @ vt[0]); span = proj[-1]-proj[0]
    if span < 1e-3: return 0.0, float(len(xs))
    gaps = np.diff(proj); big = gaps[gaps > (span/len(proj))*3]
    return float(big.sum()/span), float(span)

import pandas as _pd
rows=[]
for st,pbx in poison_boxes_by_stem.items():
    img=unlearn_imgs[st]
    for b in pbx:
        p=crop_gray(img,b); gf,ln=dashedness(p)
        w=float(b[2]-b[0]); h=float(b[3]-b[1])
        rows.append(dict(gap=gf,length=ln,md=max(w,h),area=w*h,bright=float(p.mean()),pmax=float(p.max())))
PS=_pd.DataFrame(rows)
print("20 poison-box stats:"); print(PS.describe().loc[["mean","std","min","50%","max"]].round(2))
GAP_MU=float(PS.gap.mean()); LEN_MU=float(PS.length.mean()); LEN_SD=float(PS.length.std())
print(f"poison dashedness gap_mu={GAP_MU:.3f} | len_mu={LEN_MU:.1f}+-{LEN_SD:.1f}")


20 poison-box stats:
       gap  length     md     area  bright    pmax
mean  0.36   35.63  42.90  1011.11   48.54  241.04
std   0.24   11.43  13.07   584.38    3.99   23.50
min   0.00   13.15  19.54   167.85   41.25  181.49
50%   0.43   37.83  48.18   784.75   48.14  255.00
max   0.70   55.82  59.85  2235.42   56.11  255.00
poison dashedness gap_mu=0.361 | len_mu=35.6+-11.4


## 13. Synthesize dashed (poison-like) vs continuous (real-like)

In [11]:
# Sinh streak tong hop: dashed (poison-like) vs continuous (real-like). Nen = nhieu giong test 16-bit.
rng=np.random.default_rng(SEED)
def bg_noise(mu=8.0,sd=4.0): return np.clip(rng.normal(mu,sd,(IMG_H,IMG_W)).astype(np.float32),0,255)
def draw_streak(cv,cx,cy,length,angle,dashed,dash_period,bright,thick=1.6):
    n=int(length)
    for t in range(-n//2,n//2):
        if dashed and (int(t+n//2)%dash_period)>=max(1,dash_period//2): continue
        x=cx+t*np.cos(angle); y=cy+t*np.sin(angle); xi,yi=int(round(x)),int(round(y))
        for dx in range(-2,3):
            for dy in range(-2,3):
                xx,yy=xi+dx,yi+dy
                if 0<=xx<IMG_W and 0<=yy<IMG_H:
                    cv[yy,xx]=min(255.0, cv[yy,xx]+bright*np.exp(-(dx*dx+dy*dy)/(2*thick*thick)))
    return cv
def make_sample(dashed):
    cv=bg_noise()
    L=float(np.clip(rng.normal(max(LEN_MU,24),max(LEN_SD,6)),12,90))
    ang=rng.uniform(0,np.pi); bright=float(np.clip(rng.normal(60,20),25,140))
    dp=int(np.clip(rng.normal(4,1.5),2,8))
    cx,cy=rng.uniform(120,IMG_W-120),rng.uniform(120,IMG_H-120)
    cv=draw_streak(cv,cx,cy,L,ang,dashed,dp,bright)
    cv=np.clip(cv+rng.normal(0,2.0,cv.shape),0,255).astype(np.float32)  # them nhieu -> bot ly tuong
    hw=L/2; c,s=abs(np.cos(ang)),abs(np.sin(ang))
    bw=max(10,2*hw*c+6); bh=max(10,2*hw*s+6)
    box=np.array([cx-bw/2,cy-bh/2,cx+bw/2,cy+bh/2],np.float32)
    return np.repeat(cv[:,:,None],3,axis=2), box
N_SYN=250
syn_dashed=[make_sample(True) for _ in range(N_SYN)]
syn_cont  =[make_sample(False) for _ in range(N_SYN)]
print(f"synth: {N_SYN} dashed + {N_SYN} continuous")


synth: 250 dashed + 250 continuous


## 14. GATE 1 - Reconstruction fidelity (poisoned fire rate)

In [12]:
# GATE 1 (fidelity): poisoned model co ban vao synthetic dashed manh hon continuous?
def fire_at(pr,img,box,iou_thr=0.2,conf=0.2):
    out=pr(img)["instances"].to("cpu"); b=out.pred_boxes.tensor.numpy(); s=out.scores.numpy()
    k=s>=conf; b=b[k]; s=s[k]
    if len(b)==0: return 0.0
    io=iou_matrix(box[None],b)[0]; j=io.argmax()
    return float(s[j]) if io[j]>=iou_thr else 0.0
fd=np.array([fire_at(poisoned_predictor,i,b) for i,b in syn_dashed])
fc=np.array([fire_at(poisoned_predictor,i,b) for i,b in syn_cont])
print(f"POISONED fire synth DASHED : rate={(fd>0).mean():.2f} mean_conf={(fd[fd>0].mean() if (fd>0).any() else 0):.3f}")
print(f"POISONED fire synth CONT   : rate={(fc>0).mean():.2f} mean_conf={(fc[fc>0].mean() if (fc>0).any() else 0):.3f}")
FIDELITY_OK=(fd>0).mean() >= max(0.3, 1.5*(fc>0).mean())
print(f"-> FIDELITY_OK={FIDELITY_OK} (dashed fire rate >> continuous = tai dung dung trigger)")


POISONED fire synth DASHED : rate=0.96 mean_conf=0.670
POISONED fire synth CONT   : rate=0.98 mean_conf=0.699
-> FIDELITY_OK=False (dashed fire rate >> continuous = tai dung dung trigger)


## 15. GATE 2 - DINOv2 separation on synthetic set

In [13]:
# GATE 2 (separation): DINOv2 bank (20 poison that) co tach synthetic dashed vs continuous?
from sklearn.metrics import roc_auc_score
emb_d=np.concatenate([embed_boxes(i,b[None]) for i,b in syn_dashed],0)
emb_c=np.concatenate([embed_boxes(i,b[None]) for i,b in syn_cont],0)
sim_d=sim_to_bank(emb_d); sim_c=sim_to_bank(emb_c)
y=np.r_[np.ones(len(sim_d)),np.zeros(len(sim_c))]
AUROC=float(roc_auc_score(y,np.r_[sim_d,sim_c]))
cohd=float((sim_d.mean()-sim_c.mean())/(np.sqrt((sim_d.var()+sim_c.var())/2)+1e-9))
band=float(np.median(sim_d)-np.median(sim_c))
print(f"DINOv2 sim: synth_dashed={sim_d.mean():.3f} synth_cont={sim_c.mean():.3f} | AUROC={AUROC:.3f} cohens_d={cohd:.3f}")
print(f"band(median dashed - median cont)={band:.3f}  [NB29 calib band chi 0.03]")
SEP_OK=AUROC>=0.90 and band>=0.05
print(f"-> SEP_OK={SEP_OK}")


DINOv2 sim: synth_dashed=0.782 synth_cont=0.735 | AUROC=0.818 cohens_d=1.291
band(median dashed - median cont)=0.045  [NB29 calib band chi 0.03]
-> SEP_OK=False


## 16. Summary & decision gate

In [14]:
print("="*60)
print("NB30 = Trigger reconstruction (dashed-streak) + fidelity/separation gates")
print(f"GATE1 fidelity : dashed_fire={(fd>0).mean():.2f} vs cont_fire={(fc>0).mean():.2f} -> {FIDELITY_OK}")
print(f"GATE2 separation: AUROC={AUROC:.3f} cohens_d={cohd:.3f} band={band:.3f} -> {SEP_OK}")
print("-"*60)
if FIDELITY_OK and SEP_OK:
    print("CA HAI GATE DAT -> synthetic set dang tin. Buoc sau (NB31):")
    print(" (a) forget-set day: dashed=forget (empty GT), continuous=retain (teacher GT) -> unlearn co dinh huong.")
    print(" (b) HOAC train discriminator calib ON-MANIFOLD tu synthetic (thoat band 0.03 cua 20 box le).")
elif not FIDELITY_OK:
    print("FIDELITY THAP -> trigger khong phai dashed thuan. Can inversion gradient-based (DISTIL): toi uu patch de max poisoned logit.")
else:
    print("FIDELITY OK nhung SEP thap -> DINOv2 khong tach duoc synthetic. Doi discriminator (CNN train tren synthetic co nhan).")
print("CANH BAO (bai hoc Smooth-Cactus0 nb03): synthetic co the qua ly tuong -> PHAI cross-validate tren real test dets truoc khi tin va nop.")


NB30 = Trigger reconstruction (dashed-streak) + fidelity/separation gates
GATE1 fidelity : dashed_fire=0.96 vs cont_fire=0.98 -> False
GATE2 separation: AUROC=0.818 cohens_d=1.291 band=0.045 -> False
------------------------------------------------------------
FIDELITY THAP -> trigger khong phai dashed thuan. Can inversion gradient-based (DISTIL): toi uu patch de max poisoned logit.
CANH BAO (bai hoc Smooth-Cactus0 nb03): synthetic co the qua ly tuong -> PHAI cross-validate tren real test dets truoc khi tin va nop.
